In [1]:
import os
import json
from glob import glob
from pathlib import Path
from urllib.parse import urlparse

import httpx
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset


ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

/home/octoopt/workspace/projects/personal/data_enrichment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def init_login():
    load_dotenv()
    login(os.getenv("HUGGINGFACE_TOKEN"))
    print("Login successful")

In [ ]:
import os
import httpx
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux i686; rv:109.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/121.0",
]

def download_image(image_url, save_path, timeout=10):
    """
    Download an image from a URL and save it to the specified path using httpx.
    """
    # try:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
        "Connection": "keep-alive",
    }

    with httpx.Client(follow_redirects=True) as client:
        response = client.get(image_url, headers=headers, timeout=timeout)
        response.raise_for_status()  # raise error if 403/404 etc.

        # Create directory if needed
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        with open(save_path, "wb") as f:
            f.write(response.content)

    return True

    # except httpx.HTTPStatusError as e:
    #     print(f"❌ HTTP error {e.response.status_code} for {image_url}")
    # except Exception as e:
    #     print(f"❌ Failed to download {image_url}: {e}")

    return False


def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None


def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False

## Old days 

In [8]:
DATASET_DIR = DATA_DIR / "wiki_thai_people_artist.json"

In [9]:
ds = pd.read_json(str(DATASET_DIR))
ds.head()

,profile_url,name,gender,image_url,metadata,description
0,https://en.wikipedia.org/wiki/4Eve,,,https://upload.wikimedia.org/wikipedia/commons...,"{'Origin': 'Bangkok, Thailand', 'Genres': 'T-p...",is a Thai girl group formed in 2020. The group...
1,https://en.wikipedia.org/wiki/Four-Mod,,,https://upload.wikimedia.org/wikipedia/commons...,"{'Origin': 'Bangkok , Thailand', 'Genres': 'Po...",(โฟร์-มด) was a duo which formed in 2005. Th...
2,https://en.wikipedia.org/wiki/Fonthip_Watchara...,Fonthip Watcharatrakul,,https://upload.wikimedia.org/wikipedia/commons...,"{'Born': '( 1990-07-19 ) July 19, 1990 (age 35...","( : ,"
3,https://en.wikipedia.org/wiki/Florence_Faivre,Florence Faivre,,https://upload.wikimedia.org/wikipedia/commons...,{'Born': 'Florence Vanida Faivre ( 1983-06-08 ...,( :
4,https://en.wikipedia.org/wiki/Sarocha_Chankimha,Sarocha Chankimha,,https://upload.wikimedia.org/wikipedia/commons...,{'Born': 'Sarocha Chankimha ( 1998-08-08 ) 8 A...,( :


In [10]:
ds = ds.drop(columns=["image_url"])
ds.head()

,name,url,summary,categories,timestamp
0,Sacheus !Gonteb,https://en.wikipedia.org/wiki/Sacheus_!Gonteb,Rear Admiral Sacheus Randy !Gonteb is a Namibi...,"[Articles with short description, Living peopl...",2025-08-02 04:53:49.485904
1,1.Cuz,https://en.wikipedia.org/wiki/1.Cuz,"Abas Abdikarim Bakar, better known as 1.Cuz (b...","[1997 births, 21st-century male rappers, Artic...",2025-08-02 04:53:51.707057
2,1da Banton,https://en.wikipedia.org/wiki/1da_Banton,"Godson Ominibie Epelle, professionally known a...","[1994 births, 21st-century Nigerian musicians,...",2025-08-02 04:53:54.208175
3,1nonly,https://en.wikipedia.org/wiki/1nonly,"Nathan Scott Fuller (born April 6, 2004), bett...","[2004 births, 21st-century American male rappe...",2025-08-02 04:53:56.621336
4,1ucid,https://en.wikipedia.org/wiki/1ucid,"Kwadwo Bedihene, known professionally as 1ucid...","[20th-century births, Afrobeats musicians, All...",2025-08-02 04:53:59.252879


In [12]:
hf_ds = Dataset.from_pandas(df=ds)
hf_ds

Dataset({
    features: ['name', 'url', 'summary', 'categories', 'timestamp'],
    num_rows: 1300
})

In [14]:
data_id = "minhleduc/wiki_famous_person_00"

hf_ds.push_to_hub(data_id, commit_message="Initial commit")

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.07s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/minhleduc/wiki_famous_person_00/commit/1fd688c7abaa1f6cbf506a834c2a75cbacca7549', commit_message='Initial commit', commit_description='', oid='1fd688c7abaa1f6cbf506a834c2a75cbacca7549', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/minhleduc/wiki_famous_person_00', endpoint='https://huggingface.co', repo_type='dataset', repo_id='minhleduc/wiki_famous_person_00'), pr_revision=None, pr_num=None)

## 02.10.2025

In [ ]:
datafile = str(DATA_DIR / "raw" / "22102025" / ".json")
json_data = read_json(datafile)

In [5]:
# datafile = str(DATA_DIR / "german" / "politicians")
datafile = str(DATA_DIR / "wiki_other_nations")
datapaths = glob(datafile + "/**.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer_002.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer_003.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer_001.json']

In [6]:
politician_profiles = []

for path in datapaths:
    json_data = read_json(file_path=path)
    politician_profiles += json_data


len(politician_profiles)

6167

In [7]:
politician_profiles[10]

{'title': 'Lauri Ylönen',
 'url': 'https://en.wikipedia.org/wiki/Lauri_Yl%C3%B6nen',
 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/ce/The_Rasmus_-_2025250_125352_2025-09-07_ZDF-Fernsehgarten_-_Sven_-_5DS_R_-_0115_-_5DSR3551_%28cropped%3B_Lauri_Yl%C3%B6nen%29.jpg/250px-The_Rasmus_-_2025250_125352_2025-09-07_ZDF-Fernsehgarten_-_Sven_-_5DS_R_-_0115_-_5DSR3551_%28cropped%3B_Lauri_Yl%C3%B6nen%29.jpg',
 'summary': 'Lauri Johannes Ylönen (born 23 April 1979) is a Finnish singer-songwriter, best known as the co-founder and frontman of the Finnish alternative rock band The Rasmus .',
 'infobox': {'Born': '( 1979-04-23 ) 23 April 1979 (age\xa046) Helsinki , Finland',
  'Occupation': 'Singer/songwriter',
  'Instrument(s)': 'Vocals , guitar , piano',
  'Labels': 'Playground Music Universal Music ( Finland )',
  'Website': 'amanda.fm'}}

In [4]:
write_json(data=politician_profiles, 
file_path=DATA_DIR / "wiki_other_nations" / "other_country_singer.json")

NameError: name 'politician_profiles' is not defined

In [6]:
count = 0

save_dir = str(DATA_DIR / "images" / "other_nation_wiki")

for idx in tqdm(range(len(politician_profiles))):
    data = politician_profiles[idx]
    image_url = data["image_url"]
    if not image_url:
        continue
    name = data["title"].replace(" ", "_")
    local_path = f"{save_dir}/{name}.jpg"
    data['local_path'] = local_path
    download_image(image_url, local_path)

NameError: name 'politician_profiles' is not defined

In [26]:
filepath = str(DATA_DIR / "raw" / "22102025")
datapaths = glob(filepath + "/onthisday*.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_people_001.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_gen_lost.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_genalpha.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_genx.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_gen_silent.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_genz.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_genbabyboomer.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/22102025/onthisday_gen_millenials.json']

In [19]:
def _get_name_from_url(image_url: str):
    image_name = image_url.split("/")[-1]
    name = image_name.split("-")[: -1]
    return " ".join(name).strip().title()



_get_name_from_url(image_url="https://www.thefamouspeople.com/profiles/thumbs/annette-lu-1.jpg")
    

'Annette Lu'

In [27]:
final_profiles = []

for filepath in datapaths:
    profiles = read_json(file_path=filepath)

    for profile in profiles:
        data = {}
        image_url = profile.get('image_url', '')
        if not image_url: 
            print(f"Image URL is missing for profile: {profile}")
            continue

        profile_name = profile.get('name', '')
        if not profile_name:
            profile_name = _get_name_from_url(image_url=image_url)
        profile_name = profile_name.title()

        data['name'] = profile_name
        data['mainType'] = profile.get('mainType', '')
        data['dateOfBirth'] =profile.get('dateOfBirth', '')
        data['homePlace'] =profile.get('homePlace', '')
        data['workPlace'] = profile.get('workPlace', '')

        # description = '\n\n'.join([profile.get('about', ''), profile.get('beforeFame', ''), profile.get('trivia', '')])
        description = profile.get('about', '')
        gender = profile.get('gender', '')

        if not gender: 
            if "He" in description:
                gender = "Male"
            elif "She" in description:
                gender = "Female"
            else:
                gender = ""

        data['gender'] = gender


        data['image_url'] = profile.get('image_url', '')
        data['profile_url'] =profile.get('profile_url', '')

        metadata = {
            "sunSign": profile.get('sunSign', ''),  
            'description': description
        }
        data['metadata'] = metadata
        final_profiles.append(data)

len(final_profiles)

Image URL is missing for profile: {'profile_url': '', 'name': '', 'mainType': '', 'image_url': '', 'homePlace': '', 'gender': '', 'dateOfBirth': '', 'nationality': '', 'about': ''}


3592

In [ ]:
# final_profiles = []
# DATASET_DIR = DATA_DIR / "thailand_profiles.json"
# profiles = read_json(file_path=str(DATASET_DIR))

# for profile in profiles:
#     data = {}
#     profile_name = profile.get('name', '').title()
#     image_url = profile.get('image_url', '')
#     metadata = profile.get('metadata', {})
#     if not profile_name:
#         image_name = image_url.split('/')[-1].split('.')[0]
#         image_name_components = image_name.split('-')[:-1]
#         profile_name = ' '.join(image_name_components).title()
#     data['name'] = profile_name
#     data['mainType'] = metadata.get('Occupations', '')
#     data['dateOfBirth'] =metadata.get('Born', '')
#     data['homePlace'] =metadata.get('Labels', '')
#     data['workPlace'] = metadata.get('workPlace', 'Thailand')
#     data['gender'] =profile.get('gender', '')


#     data['image_url'] = profile.get('image_url', '')
#     data['profile_url'] =profile.get('profile_url', '')
#     data['metadata'] = metadata

#     final_profiles.append(data)

# len(final_profiles)

200

In [30]:
filepath = str(DATA_DIR)
datapaths = glob(filepath + "/20252210*.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/20252210_001_fmppl_bf.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/20252210_003_fmppl_bf.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/20252210_002_fmppl_bf.json']

In [31]:
merged_profiles = []

for filepath in datapaths:
    profiles = read_json(filepath)
    merged_profiles += profiles

len(merged_profiles)

8920

In [ ]:
write_json(
    data=final_profiles, 
    # data=merged_profiles, 
        file_path='../data/20252210_mrgdprf_bf.json', 
)

✓ Written JSON to: ../data/20252210_mrgdprf_bf.json


True

In [40]:
final_profiles = read_json(file_path=str(DATA_DIR / "20252210_mrgdprf_bf.json"))

len(final_profiles)

8920

In [41]:
"""
Only get profile that get image
"""

usable_profiles = []
save_dir = str(DATA_DIR / "images" / "20252210_mrgdprf")


for idx in tqdm(range(len(final_profiles))):
    try:
        profile = final_profiles[idx]
        image_url = profile['image_url']
        img_name = profile["name"].replace(" ", "_")
        local_path = f"{save_dir}/{img_name}.jpg"
        profile['local_path'] = local_path
        is_success = download_image(image_url, local_path)
        if is_success:
            usable_profiles.append(profile)

    except Exception as e:
        print(e)
        continue

  0%|          | 1/8920 [00:01<2:53:44,  1.17s/it]

✅ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/20252210_mrgdprf/Charli_Damelio.jpg


  0%|          | 2/8920 [00:01<1:42:51,  1.45it/s]

✅ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/20252210_mrgdprf/Khabane_Lame.jpg


  0%|          | 3/8920 [00:02<1:58:18,  1.26it/s]

...

✅ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/20252210_mrgdprf/Lily-Rose_Depp.jpg
✅ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/20252210_mrgdprf/Kylian_Mbappé.jpg


100%|█████████▉| 8919/8920 [2:02:52<00:00,  2.72it/s]

✅ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/20252210_mrgdprf/Lev_Cameron.jpg


100%|██████████| 8920/8920 [2:02:53<00:00,  1.21it/s]

✅ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/20252210_mrgdprf/Emma_Watson.jpg


In [42]:
write_json(
    data=usable_profiles, 
    file_path='../data/20252210_mrgdprf_aftr.json', 
)

✓ Written JSON to: ../data/20252210_mrgdprf_aftr.json


True